# Data Integration

## Objective
To combine the cleaned Customers, Products, Orders, and Tickets datasets using their common identifiers and create a unified dataset for further analysis and machine learning.

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
customers = pd.read_csv("customers_cleaned.csv")
print("Customers Dataset Loaded:", customers.shape)
products = pd.read_csv("products_cleaned.csv")
print("Products Dataset Loaded:", products.shape)
orders = pd.read_csv("orders_cleaned.csv")
print("Orders Dataset Loaded:", orders.shape)
tickets = pd.read_csv("tickets_cleaned.csv")
print("Tickets Dataset Loaded:", tickets.shape)

Customers Dataset Loaded: (1200, 5)
Products Dataset Loaded: (40, 4)
Orders Dataset Loaded: (12965, 9)
Tickets Dataset Loaded: (164, 5)


In [3]:
print("CUSTOMERS")
print(customers.columns.tolist())

print("\nPRODUCTS")
print(products.columns.tolist())

print("\nORDERS")
print(orders.columns.tolist())

print("\nTICKETS")
print(tickets.columns.tolist())

CUSTOMERS
['customer_id', 'city', 'signup_date', 'age', 'gender']

PRODUCTS
['product_id', 'product_name', 'category', 'price_inr']

ORDERS
['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes']

TICKETS
['ticket_id', 'customer_id', 'order_id', 'text', 'true_category']


Check Key Uniqueness

In [4]:
print("CUSTOMERS")
print("Duplicate customer IDs:",
      customers["customer_id"].duplicated().sum())

print("\nPRODUCTS")
print("Duplicate product IDs:",
      products["product_id"].duplicated().sum())

print("\nORDERS")
print("Duplicate order IDs:",
      orders["order_id"].duplicated().sum())

print("\nTICKETS")
print("Duplicate ticket IDs:",
      tickets["ticket_id"].duplicated().sum())

CUSTOMERS
Duplicate customer IDs: 0

PRODUCTS
Duplicate product IDs: 0

ORDERS
Duplicate order IDs: 656

TICKETS
Duplicate ticket IDs: 0


First Integration: Customers + Orders

In [5]:
customer_orders = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="left"
)

print("Integrated Shape:", customer_orders.shape)
print("\nMissing Customer Details:")
print(customer_orders["city"].isnull().sum())

print("\nPreview:")
display(customer_orders.head())

Integrated Shape: (12965, 13)

Missing Customer Details:
0

Preview:


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes,city,signup_date,age,gender
0,ORD-108250,C0792,P004,2,2025-09-01 12:24:59,completed,0,310.00,17.0,Pune,2024-11-17,40.0,Male
1,ORD-101939,C0190,P003,1,2025-08-17 07:07:23,completed,5,232.75,21.0,Pune,2025-02-25,29.0,Male
2,ORD-108765,C0841,P033,1,2025-07-13 09:42:32,completed,10,1349.10,27.0,Delhi,2024-10-13,20.0,Female
3,ORD-108861,C0852,P010,1,2025-07-04 11:23:24,completed,0,45.00,30.0,Chennai,2024-06-15,54.0,Female
4,ORD-109150,C0877,P018,2,2025-05-07 15:35:01,completed,20,96.00,28.0,Hyderabad,2025-01-18,23.0,Male


Integrate Products

In [6]:
integrated_data = pd.merge(
    customer_orders,
    products,
    on="product_id",
    how="left"
)

print("Integrated Shape:", integrated_data.shape)

print("\nMissing Product Details:")
print(integrated_data["product_name"].isnull().sum())

print("\nPreview:")
display(integrated_data.head())

Integrated Shape: (12965, 16)

Missing Product Details:
0

Preview:


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes,city,signup_date,age,gender,product_name,category,price_inr
0,ORD-108250,C0792,P004,2,2025-09-01 12:24:59,completed,0,310.00,17.0,Pune,2024-11-17,40.0,Male,Sunflower Oil 1L,Grocery,155
1,ORD-101939,C0190,P003,1,2025-08-17 07:07:23,completed,5,232.75,21.0,Pune,2025-02-25,29.0,Male,Atta 5kg,Grocery,245
2,ORD-108765,C0841,P033,1,2025-07-13 09:42:32,completed,10,1349.10,27.0,Delhi,2024-10-13,20.0,Female,Wireless Earbuds,Electronics,1499
3,ORD-108861,C0852,P010,1,2025-07-04 11:23:24,completed,0,45.00,30.0,Chennai,2024-06-15,54.0,Female,Curd 400g,Dairy,45
4,ORD-109150,C0877,P018,2,2025-05-07 15:35:01,completed,20,96.00,28.0,Hyderabad,2025-01-18,23.0,Male,Namkeen 200g,Snacks,60


Prepare Tickets Integration

In [7]:
# Get all non-missing order IDs from Tickets
ticket_order_ids = tickets["order_id"].dropna().unique()

# Find matching order records
ticket_order_history = orders[
    orders["order_id"].isin(ticket_order_ids)
]

# Analyze repeated order history
ticket_order_check = (
    ticket_order_history
    .groupby("order_id")
    .agg(
        history_rows=("order_id", "size"),
        customers=("customer_id", "nunique"),
        products=("product_id", "nunique"),
        statuses=("status", "nunique")
    )
    .reset_index()
)

print("Tickets with valid Order IDs:",
      tickets["order_id"].notna().sum())

print("Matching Order IDs:",
      tickets["order_id"].dropna().isin(
          orders["order_id"]
      ).sum())

print("\nOrder History Summary:")
display(ticket_order_check.head(10))

print("\nOrder IDs with Multiple History Rows:",
      (ticket_order_check["history_rows"] > 1).sum())

print("Order IDs with Multiple Products:",
      (ticket_order_check["products"] > 1).sum())

Tickets with valid Order IDs: 156
Matching Order IDs: 156

Order History Summary:


,order_id,history_rows,customers,products,statuses
0,ORD-100142,1,1,1,1
1,ORD-100227,1,1,1,1
2,ORD-100246,1,1,1,1
3,ORD-100455,1,1,1,1
4,ORD-100548,1,1,1,1
5,ORD-100640,1,1,1,1
6,ORD-100782,1,1,1,1
7,ORD-100789,1,1,1,1
8,ORD-100816,1,1,1,1
9,ORD-100855,1,1,1,1



Order IDs with Multiple History Rows: 5
Order IDs with Multiple Products: 0


Inspect the 5 Repeated Order Histories

In [8]:
multiple_order_ids = ticket_order_check.loc[
    ticket_order_check["history_rows"] > 1,
    "order_id"
]

repeated_order_history = orders[
    orders["order_id"].isin(multiple_order_ids)
].sort_values(["order_id", "order_ts"])

display(
    repeated_order_history[
        [
            "order_id",
            "customer_id",
            "product_id",
            "order_ts",
            "status",
            "order_amount",
            "delivery_minutes"
        ]
    ]
)

,order_id,customer_id,product_id,order_ts,status,order_amount,delivery_minutes
1249,ORD-103449,C0324,P036,2025-05-17 06:57:10,pending,1438.20,NaN
10971,ORD-103449,C0324,P036,2025-05-17 07:00:10,completed,1438.20,8.0
3687,ORD-103995,C0376,P027,2025-04-02 09:41:14,pending,93.10,NaN
7507,ORD-103995,C0376,P027,2025-04-02 09:44:14,completed,93.10,11.0
609,ORD-105103,C0491,P009,2025-09-17 22:47:35,pending,21.75,NaN
5565,ORD-105103,C0491,P009,2025-09-17 22:57:35,completed,21.75,28.0
1541,ORD-105355,C0521,P039,2025-09-21 18:30:54,pending,261.75,NaN
3247,ORD-105355,C0521,P039,2025-09-21 18:32:54,completed,261.75,73.0
4041,ORD-109516,C0910,P039,2025-04-07 12:22:15,pending,523.50,NaN
8483,ORD-109516,C0910,P039,2025-04-07 12:31:15,completed,523.50,18.0


Create Latest Order Snapshot

In [9]:
# Sort orders by timestamp
orders_sorted = orders.sort_values("order_ts")

# Keep the latest record for each order
orders_latest = orders_sorted.drop_duplicates(
    subset="order_id",
    keep="last"
).copy()

print("Original Orders:", orders.shape)
print("Latest Order Snapshot:", orders_latest.shape)

print(
    "Duplicate Order IDs:",
    orders_latest["order_id"].duplicated().sum()
)

print("\nRepeated Order History Check:")
display(
    orders_latest[
        orders_latest["order_id"].isin(multiple_order_ids)
    ][
        ["order_id", "order_ts", "status", "delivery_minutes"]
    ]
)

Original Orders: (12965, 9)
Latest Order Snapshot: (12309, 9)
Duplicate Order IDs: 0

Repeated Order History Check:


,order_id,order_ts,status,delivery_minutes
7507,ORD-103995,2025-04-02 09:44:14,completed,11.0
8483,ORD-109516,2025-04-07 12:31:15,completed,18.0
10971,ORD-103449,2025-05-17 07:00:10,completed,8.0
5565,ORD-105103,2025-09-17 22:57:35,completed,28.0
3247,ORD-105355,2025-09-21 18:32:54,completed,73.0


Add product details to latest orders

In [10]:
latest_order_data = pd.merge(
    orders_latest,
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("Latest Order Data Shape:", latest_order_data.shape)

print("\nMissing Product Names:")
print(latest_order_data["product_name"].isnull().sum())

print("\nDuplicate Order IDs:")
print(latest_order_data["order_id"].duplicated().sum())

display(latest_order_data.head())

Latest Order Data Shape: (12309, 12)

Missing Product Names:
0

Duplicate Order IDs:
0


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes,product_name,category,price_inr
0,ORD-110093,C0967,P020,1,2025-04-01 07:08:28,completed,15,127.50,19.0,Dark Chocolate 80g,Snacks,150
1,ORD-111396,C1117,P036,2,2025-04-01 07:11:59,completed,0,1598.00,25.0,Phone Charger 20W,Electronics,799
2,ORD-102006,C0194,P009,1,2025-04-01 07:12:43,cancelled,25,21.75,NaN,Toned Milk 500ml,Dairy,29
3,ORD-105921,C0569,P005,2,2025-04-01 07:54:43,completed,10,86.40,33.0,Sugar 1kg,Grocery,48
4,ORD-108115,C0776,P012,2,2025-04-01 08:02:58,completed,10,104.40,21.0,Butter 100g,Dairy,58


Integrate Customers with Tickets

In [11]:
tickets_customers = pd.merge(
    tickets,
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Tickets + Customers Shape:", tickets_customers.shape)

print("\nMissing Customer City:")
print(tickets_customers["city"].isnull().sum())

print("\nOriginal Ticket Rows:", tickets.shape[0])
print("Integrated Ticket Rows:", tickets_customers.shape[0])

display(tickets_customers.head())

Tickets + Customers Shape: (164, 9)

Missing Customer City:
0

Original Ticket Rows: 164
Integrated Ticket Rows: 164


,ticket_id,customer_id,order_id,text,true_category,city,signup_date,age,gender
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud,Chennai,2024-03-13,28.0,Female
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing,Mumbai,2024-10-28,51.0,Male
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical,Pune,2025-01-16,18.0,Female
3,T0159,C1172,NaN,NaN,NaN,Delhi,2024-08-28,27.0,Male
4,T0080,C0910,ORD-109516,I was charged for order ORD-109516 that I neve...,fraud,Pune,2024-01-13,54.0,Female


Add latest order details to tickets

In [12]:
tickets_integrated = pd.merge(
    tickets_customers,
    latest_order_data,
    on=["order_id", "customer_id"],
    how="left",
    validate="many_to_one"
)

print("Final Tickets Integrated Shape:", tickets_integrated.shape)

print("\nMissing Order Status:")
print(tickets_integrated["status"].isnull().sum())

print("\nTicket Rows:", tickets.shape[0])
print("Integrated Rows:", tickets_integrated.shape[0])

print("\nDuplicate Ticket IDs:")
print(tickets_integrated["ticket_id"].duplicated().sum())

display(tickets_integrated.head())

Final Tickets Integrated Shape: (164, 19)

Missing Order Status:
8

Ticket Rows: 164
Integrated Rows: 164

Duplicate Ticket IDs:
0


,ticket_id,customer_id,order_id,text,true_category,city,signup_date,age,gender,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes,product_name,category,price_inr
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud,Chennai,2024-03-13,28.0,Female,P031,1.0,2025-06-27 20:19:39,completed,25.0,165.00,27.0,Deodorant 150ml,Personal Care,220.0
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing,Mumbai,2024-10-28,51.0,Male,P001,1.0,2025-08-15 10:31:33,completed,15.0,109.65,44.0,Basmati Rice 1kg,Grocery,129.0
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical,Pune,2025-01-16,18.0,Female,P012,2.0,2025-06-23 18:37:44,completed,10.0,104.40,34.0,Butter 100g,Dairy,58.0
3,T0159,C1172,NaN,NaN,NaN,Delhi,2024-08-28,27.0,Male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T0080,C0910,ORD-109516,I was charged for order ORD-109516 that I neve...,fraud,Pune,2024-01-13,54.0,Female,P039,2.0,2025-04-07 12:31:15,completed,25.0,523.50,18.0,Phone Stand,Electronics,349.0


Final integration validation

In [13]:
print("Total Tickets:", len(tickets_integrated))

print(
    "Tickets with Order ID:",
    tickets_integrated["order_id"].notna().sum()
)

print(
    "Tickets with Matched Orders:",
    tickets_integrated["status"].notna().sum()
)

print(
    "Tickets without Order ID:",
    tickets_integrated["order_id"].isna().sum()
)

print(
    "Tickets with Order ID but No Match:",
    (
        tickets_integrated["order_id"].notna()
        & tickets_integrated["status"].isna()
    ).sum()
)

print(
    "Duplicate Ticket IDs:",
    tickets_integrated["ticket_id"].duplicated().sum()
)

Total Tickets: 164
Tickets with Order ID: 156
Tickets with Matched Orders: 156
Tickets without Order ID: 8
Tickets with Order ID but No Match: 0
Duplicate Ticket IDs: 0


In [14]:
tickets_integrated.to_csv(
    "tickets_integrated.csv",
    index=False
)

print("Integrated dataset saved successfully!")

Integrated dataset saved successfully!


In [15]:
check_data = pd.read_csv("tickets_integrated.csv")

print("Saved Dataset Shape:", check_data.shape)
print("Duplicate Ticket IDs:",
      check_data["ticket_id"].duplicated().sum())

Saved Dataset Shape: (164, 19)
Duplicate Ticket IDs: 0


# Data Integration – Project Documentation

## 1. Introduction

The Data Integration stage combines the four cleaned datasets — Customers, Products, Orders, and Tickets — into meaningful datasets for further analysis and machine learning.

The main objective is to establish relationships between customers, their orders, purchased products, and support tickets while preserving data accuracy and record consistency.

## 2. Input Datasets

The integration process uses the following cleaned datasets:

- **Customers:** Contains customer information such as customer ID, city, signup date, age, and gender.
- **Products:** Contains product ID, product name, category, and price.
- **Orders:** Contains order ID, customer ID, product ID, quantity, order timestamp, order status, discount percentage, order amount, and delivery time.
- **Tickets:** Contains ticket ID, customer ID, order ID, ticket text, and actual ticket category.

All four datasets were loaded from their respective cleaned CSV files using Pandas.

## 3. Library Import and Dataset Loading

Pandas and NumPy libraries were imported for data manipulation and analysis.

The cleaned CSV files were loaded into separate DataFrames:

- customers
- products
- orders
- tickets

The initial dataset dimensions were verified to ensure that all files were loaded correctly.

## 4. Customers and Orders Integration

The Customers and Orders datasets were combined using a Left Join on the `customer_id` column.

The purpose was to enrich order records with customer information, including city, signup date, age, and gender.

All original order records were preserved during the merge.

The resulting dataset contained 12,965 rows and 13 columns.

Validation confirmed that no customer city values were missing after integration.

## 5. Products Integration

Product information was added to the customer-order dataset using a Left Join on the `product_id` column.

The integration added the following product attributes:

- product_name
- category
- price_inr

The resulting integrated dataset contained 12,965 rows and 16 columns.

Validation confirmed that all order records had matching product information and no product names were missing.

## 6. Order History Analysis

During the integration process, repeated order IDs were identified in the Orders dataset.

The original Orders dataset contained 12,965 records, with 656 repeated order ID entries after exact duplicate removal.

Further investigation showed that repeated order IDs represented order status history rather than separate purchases.

For example, some orders had an initial `pending` status followed by a later `completed` status.

Five order IDs referenced by support tickets were specifically inspected. Each contained two history records, representing a pending-to-completed status progression.

Therefore, the original order history was preserved without deleting these records.

## 7. Latest Order Snapshot Creation

To prevent repeated order IDs from multiplying ticket records during integration, a separate latest-order snapshot was created.

The Orders dataset was sorted using the `order_ts` column, and the latest record for each `order_id` was retained using `drop_duplicates()`.

The resulting latest-order snapshot contained 12,309 rows and 9 columns.

Validation confirmed:

- No duplicate order IDs remained.
- Each order was represented by its latest available record.
- The original Orders dataset remained unchanged.

This snapshot was subsequently enriched with product information.

The resulting `latest_order_data` dataset contained 12,309 rows and 12 columns.

No product names were missing, and all order IDs were unique.

## 8. Tickets and Customers Integration

The Tickets dataset was combined with the Customers dataset using a Left Join on `customer_id`.

This integration added customer information to every support ticket.

The following customer attributes were included:

- city
- signup_date
- age
- gender

The resulting `tickets_customers` dataset contained 164 rows and 9 columns.

Validation confirmed that:

- All 164 ticket records were preserved.
- No customer city values were missing.
- No ticket records were lost during integration.

This approach also ensured that tickets with missing order IDs retained their available customer information.

## 9. Tickets and Latest Orders Integration

The customer-enriched ticket dataset was combined with `latest_order_data`.

The merge was performed using both `order_id` and `customer_id` to ensure that ticket records were connected to the correct customer's order.

A Left Join was used to preserve all original ticket records.

The integration added order and product attributes, including:

- product_id
- quantity
- order_ts
- status
- discount_pct
- order_amount
- delivery_minutes
- product_name
- category
- price_inr

The final integrated dataset, `tickets_integrated`, contained 164 rows and 19 columns.

## 10. Final Data Validation

Several validation checks were performed to verify the accuracy and consistency of the final integrated dataset.

The final validation confirmed:

- Total ticket records: 164.
- Tickets with available order IDs: 156.
- Successfully matched orders: 156.
- Tickets without order IDs: 8.
- Tickets with an order ID but no matching order: 0.
- Duplicate ticket IDs: 0.

The eight tickets without order IDs were retained because their order references were missing in the original Tickets dataset.

Their available customer information was preserved, while unavailable order and product details remained missing.

No artificial order information was generated.

## 11. Exporting the Final Dataset

The final integrated dataset was exported as:

`tickets_integrated.csv`

The exported file was reloaded using Pandas to verify successful saving.

The final verification confirmed:

- Dataset shape: 164 rows and 19 columns.
- Duplicate ticket IDs: 0.

The exported dataset was successfully validated.

## 12. Conclusion

The Data Integration stage successfully combined customer, product, order, and support ticket information into structured datasets.

The integration process preserved the original order history, created a separate latest-order snapshot, maintained all ticket records, and ensured accurate relationships between customers, orders, and products.

The final integrated dataset provides a consolidated foundation for downstream machine learning and AI applications.

It can support customer segmentation, churn prediction, support ticket classification, automated ticket routing, and AI-powered customer support solutions.

**Final Output:** `tickets_integrated.csv`

**Final Dataset Shape:** 164 rows × 19 columns.

In [16]:
import pandas as pd
import sqlite3

# Load cleaned datasets
customers_df = pd.read_csv("../data/clean/customers.csv")
products_df = pd.read_csv("../data/clean/products.csv")
orders_df = pd.read_csv("../data/clean/orders.csv")

print("Customers:", customers_df.shape)
print("Products:", products_df.shape)
print("Orders:", orders_df.shape)

Customers: (1200, 5)
Products: (40, 4)
Orders: (12309, 9)


In [17]:
import os
import sqlite3

# Create database folder
os.makedirs("../data", exist_ok=True)

# Connect to SQLite database
conn = sqlite3.connect("../data/instamart.db")

# Save datasets as SQL tables
customers_df.to_sql("customers", conn, if_exists="replace", index=False)
products_df.to_sql("products", conn, if_exists="replace", index=False)
orders_df.to_sql("orders", conn, if_exists="replace", index=False)

print("SQLite database created successfully!")

SQLite database created successfully!


In [18]:
# Check available tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print("Database Tables:")
print(tables)

# Check row counts
for table in ["customers", "products", "orders"]:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS total FROM {table}",
        conn
    )
    print(f"{table}: {count.iloc[0, 0]} rows")

Database Tables:
        name
0  customers
1   products
2     orders
customers: 1200 rows
products: 40 rows
orders: 12309 rows


In [19]:
# Check SQLite column data types
schema = pd.read_sql_query(
    "PRAGMA table_info(orders);",
    conn
)

print(schema[["name", "type"]])

# Check sample order timestamps
sample = pd.read_sql_query(
    "SELECT order_ts FROM orders LIMIT 5;",
    conn
)

print("\nSample Order Timestamps:")
print(sample)

               name     type
0          order_id     TEXT
1       customer_id     TEXT
2        product_id     TEXT
3          quantity  INTEGER
4          order_ts     TEXT
5            status     TEXT
6      discount_pct  INTEGER
7      order_amount     REAL
8  delivery_minutes     REAL

Sample Order Timestamps:
              order_ts
0  2025-04-01 07:08:28
1  2025-04-01 07:11:59
2  2025-04-01 07:12:43
3  2025-04-01 07:54:43
4  2025-04-01 08:02:58
